# PDD Mobility Scanner — Depth Pro Video Test

Generate metric depth maps from video using [Apple Depth Pro](https://github.com/apple/ml-depth-pro).

**How to use:**
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Run all cells
3. Upload an `.mp4` video when prompted
4. View per-frame depth maps and download the output video

Depth Pro produces sharp, metric (meters) depth maps at 2.25 megapixels without needing camera intrinsics.

## Step 1: Install Depth Pro & dependencies

In [ ]:
!pip install -q git+https://github.com/apple/ml-depth-pro.git
!pip install -q opencv-python-headless pillow_heif

## Step 2: Download pretrained weights

In [ ]:
import os

os.makedirs("checkpoints", exist_ok=True)
if not os.path.exists("checkpoints/depth_pro.pt"):
    !wget -q --show-progress https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt -P checkpoints
    print("Model downloaded.")
else:
    print("Model already present.")

## Step 3: Load model

In [ ]:
import torch
import depth_pro

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model, transform = depth_pro.create_model_and_transforms(
    device=device,
    precision=torch.float16 if device.type == "cuda" else torch.float32,
)
model.eval()
print("Depth Pro loaded.")

## Step 4: Upload video

In [ ]:
from google.colab import files

uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"Uploaded: {video_path}")

## Step 5: Extract frames

Extracts one frame per second by default. Adjust `frame_interval_sec` to change sampling rate.

In [ ]:
import cv2
from PIL import Image
import numpy as np

frame_interval_sec = 1  # @param {type:"number"}

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps
frame_step = max(1, int(fps * frame_interval_sec))

print(f"Video: {fps:.1f} FPS, {total_frames} frames, {duration:.1f}s duration")
print(f"Extracting 1 frame every {frame_interval_sec}s ({total_frames // frame_step} frames)")

frames = []
frame_indices = []
idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if idx % frame_step == 0:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(rgb)
        frame_indices.append(idx)
    idx += 1

cap.release()
print(f"Extracted {len(frames)} frames.")

## Step 6: Run Depth Pro on each frame

In [ ]:
import time

depth_maps = []

for i, rgb in enumerate(frames):
    t0 = time.time()

    # Convert numpy array to PIL and apply transform
    pil_img = Image.fromarray(rgb)
    img_tensor = transform(pil_img).to(device)

    with torch.no_grad():
        prediction = model.infer(img_tensor)

    depth = prediction["depth"].cpu().numpy()  # depth in meters
    depth_maps.append(depth)

    elapsed = time.time() - t0
    timestamp = frame_indices[i] / fps
    print(f"  Frame {i+1}/{len(frames)} (t={timestamp:.1f}s) — "
          f"depth range: {depth.min():.2f}–{depth.max():.2f} m, "
          f"{elapsed:.2f}s")

print(f"\nDone. Processed {len(depth_maps)} frames.")

## Step 7: Visualize results

Side-by-side: original frame (left) and depth map (right).

In [ ]:
import matplotlib.pyplot as plt

# Show up to 8 evenly-spaced frames
n_show = min(8, len(frames))
show_indices = np.linspace(0, len(frames) - 1, n_show, dtype=int)

fig, axes = plt.subplots(n_show, 2, figsize=(14, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for row, fi in enumerate(show_indices):
    timestamp = frame_indices[fi] / fps

    axes[row, 0].imshow(frames[fi])
    axes[row, 0].set_title(f"Frame @ {timestamp:.1f}s")
    axes[row, 0].axis("off")

    im = axes[row, 1].imshow(depth_maps[fi], cmap="turbo")
    axes[row, 1].set_title(
        f"Depth — {depth_maps[fi].min():.1f}–{depth_maps[fi].max():.1f} m"
    )
    axes[row, 1].axis("off")
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046, pad=0.04, label="meters")

plt.tight_layout()
plt.show()

## Step 8: Export side-by-side depth video

In [ ]:
import matplotlib.cm as cm

output_path = "depth_output.mp4"

# Use the first frame to determine output dimensions
h, w = frames[0].shape[:2]
out_w = w * 2  # side-by-side

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_fps = 1.0 / frame_interval_sec  # match the extraction rate
writer = cv2.VideoWriter(output_path, fourcc, out_fps, (out_w, h))

# Global depth range for consistent colormap
global_min = min(d.min() for d in depth_maps)
global_max = max(d.max() for d in depth_maps)

for rgb, depth in zip(frames, depth_maps):
    # Normalize depth to 0-1 and apply turbo colormap
    depth_resized = cv2.resize(depth, (w, h), interpolation=cv2.INTER_LINEAR)
    depth_norm = (depth_resized - global_min) / (global_max - global_min + 1e-8)
    depth_color = (cm.turbo(depth_norm)[:, :, :3] * 255).astype(np.uint8)

    # Combine side-by-side and convert RGB→BGR for OpenCV
    combined = np.concatenate([rgb, depth_color], axis=1)
    writer.write(cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))

writer.release()

print(f"Saved {output_path} ({len(frames)} frames at {out_fps} FPS)")
print(f"Global depth range: {global_min:.2f}–{global_max:.2f} m")
files.download(output_path)

## Step 9: Per-frame depth statistics

In [ ]:
import pandas as pd

stats = []
for i, depth in enumerate(depth_maps):
    stats.append({
        "frame": i,
        "timestamp_s": frame_indices[i] / fps,
        "depth_min_m": depth.min(),
        "depth_max_m": depth.max(),
        "depth_mean_m": depth.mean(),
        "depth_median_m": np.median(depth),
        "depth_std_m": depth.std(),
    })

df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))

# Plot depth stats over time
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(df_stats["timestamp_s"], df_stats["depth_min_m"],
                df_stats["depth_max_m"], alpha=0.2, label="min–max range")
ax.plot(df_stats["timestamp_s"], df_stats["depth_mean_m"], "o-", label="mean depth")
ax.plot(df_stats["timestamp_s"], df_stats["depth_median_m"], "s--", label="median depth")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Depth (m)")
ax.set_title("Depth statistics over time")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Export
df_stats.to_csv("depth_stats.csv", index=False)
files.download("depth_stats.csv")
print("Exported depth_stats.csv")